In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from openai import OpenAI
from mistralai import Mistral
import dotenv

from utils import load_json


dotenv.load_dotenv()


True

In [3]:
dev_data_path = "data/source/mrbench_v3_devset.json"
test_data_path = "data/source/mrbench_v3_testset.json"

In [8]:
dev_data = load_json(dev_data_path)
test_data = load_json(test_data_path)

len(dev_data), len(test_data)

(300, 191)

In [7]:
dev_data[0]

{'conversation_id': '221-362eb11a-f190-42a6-b2a4-985fafdcfa9e',
 'conversation_history': 'Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.\xa0\xa0Each sandwich required 1 pound each of meat and cheese and would serve 4 people.\xa0\xa0There would be 20 people in total watching the game.\xa0\xa0The meat cost $7.00 per pound and the cheese cost $3.00 per pound.\xa0\xa0How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people?\xa0\n\xa0Student: To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.\nEach sandwich requires 1+1 = 2 pounds of meat and cheese.\nFor 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.\nThe cost of 10 pounds of meat is 10 x $7.00 = $70.\nThe cost of 10 pounds of cheese is 10 x $3.00 = $30.\nThe total cost of meat and cheese is $70 + $30 = $100.\n\xa0100\xa0\n\xa0Tutor: How many pounds o

In [9]:
test_data[0]

{'conversation_id': '1030-adb61831-0383-4e51-a673-ab978590f69b',
 'conversation_history': 'Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.  Each sandwich required 1 pound each of meat and cheese and would serve 4 people.  There would be 20 people in total watching the game.  The meat cost $7.00 per pound and the cheese cost $3.00 per pound.  How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people? \n Student: To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.\nEach sandwich requires 1+1 = 2 pounds of meat and cheese.\nFor 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.\nThe cost of 10 pounds of meat is 10 x $7.00 = $70.\nThe cost of 10 pounds of cheese is 10 x $3.00 = $30.\nThe total cost of meat and cheese is $70 + $30 = $100.\n 100 \n Tutor: do you want to talk me through your solution \n Student

In [19]:
s = dev_data[0]
ch = s["conversation_history"]
ch

'Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.\xa0\xa0Each sandwich required 1 pound each of meat and cheese and would serve 4 people.\xa0\xa0There would be 20 people in total watching the game.\xa0\xa0The meat cost $7.00 per pound and the cheese cost $3.00 per pound.\xa0\xa0How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people?\xa0\n\xa0Student: To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.\nEach sandwich requires 1+1 = 2 pounds of meat and cheese.\nFor 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.\nThe cost of 10 pounds of meat is 10 x $7.00 = $70.\nThe cost of 10 pounds of cheese is 10 x $3.00 = $30.\nThe total cost of meat and cheese is $70 + $30 = $100.\n\xa0100\xa0\n\xa0Tutor: How many pounds of meat are needed for each sandwich?\xa0\n\xa0Student: Each sandwich requires 1 pound of 

In [20]:
# split at Tutor: and Student:
# Use regex to split while keeping the speaker tag
import re

def split_conversation(text):
    """
    Splits the conversation text into parts based on the speaker tags (Tutor: and Student:).
    Returns a list of tuples containing the speaker tag and the corresponding text.
    """
    # Use regex to split while keeping the speaker tag
    parts = re.findall(r'(Tutor:|Student:)(.*?)(?=Tutor:|Student:|$)', text, re.DOTALL)
    return parts

In [21]:
parts = split_conversation(ch)

In [22]:
parts

[('Tutor:',
  ' Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.\xa0\xa0Each sandwich required 1 pound each of meat and cheese and would serve 4 people.\xa0\xa0There would be 20 people in total watching the game.\xa0\xa0The meat cost $7.00 per pound and the cheese cost $3.00 per pound.\xa0\xa0How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people?\xa0\n\xa0'),
 ('Student:',
  ' To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.\nEach sandwich requires 1+1 = 2 pounds of meat and cheese.\nFor 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.\nThe cost of 10 pounds of meat is 10 x $7.00 = $70.\nThe cost of 10 pounds of cheese is 10 x $3.00 = $30.\nThe total cost of meat and cheese is $70 + $30 = $100.\n\xa0100\xa0\n\xa0'),
 ('Tutor:',
  ' How many pounds of meat are needed for each sandwich?\xa0\n\xa0'),
 ('St

In [23]:
dialogue = [{'speaker': speaker.strip(':'), 'text': content.strip()} for speaker, content in parts]
dialogue

[{'speaker': 'Tutor',
  'text': 'Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.\xa0\xa0Each sandwich required 1 pound each of meat and cheese and would serve 4 people.\xa0\xa0There would be 20 people in total watching the game.\xa0\xa0The meat cost $7.00 per pound and the cheese cost $3.00 per pound.\xa0\xa0How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people?'},
 {'speaker': 'Student',
  'text': 'To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.\nEach sandwich requires 1+1 = 2 pounds of meat and cheese.\nFor 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.\nThe cost of 10 pounds of meat is 10 x $7.00 = $70.\nThe cost of 10 pounds of cheese is 10 x $3.00 = $30.\nThe total cost of meat and cheese is $70 + $30 = $100.\n\xa0100'},
 {'speaker': 'Tutor',
  'text': 'How many pounds of meat are needed for ea

In [24]:
def extract_dialogue(conversation_history):
    """
    Extracts the dialogue from the conversation history string.
    Returns a list of dictionaries with 'speaker' and 'text' keys.
    """
    # Split the conversation history into parts
    parts = split_conversation(conversation_history)
    
    # Create a list of dictionaries for each part
    dialogue = [{'speaker': speaker.strip(':'), 'text': content.strip()} for speaker, content in parts]
    
    return dialogue

In [25]:
dialogue = extract_dialogue(ch)

In [26]:
dialogue

[{'speaker': 'Tutor',
  'text': 'Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.\xa0\xa0Each sandwich required 1 pound each of meat and cheese and would serve 4 people.\xa0\xa0There would be 20 people in total watching the game.\xa0\xa0The meat cost $7.00 per pound and the cheese cost $3.00 per pound.\xa0\xa0How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people?'},
 {'speaker': 'Student',
  'text': 'To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.\nEach sandwich requires 1+1 = 2 pounds of meat and cheese.\nFor 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.\nThe cost of 10 pounds of meat is 10 x $7.00 = $70.\nThe cost of 10 pounds of cheese is 10 x $3.00 = $30.\nThe total cost of meat and cheese is $70 + $30 = $100.\n\xa0100'},
 {'speaker': 'Tutor',
  'text': 'How many pounds of meat are needed for ea

In [27]:
s

{'conversation_id': '221-362eb11a-f190-42a6-b2a4-985fafdcfa9e',
 'conversation_history': 'Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.\xa0\xa0Each sandwich required 1 pound each of meat and cheese and would serve 4 people.\xa0\xa0There would be 20 people in total watching the game.\xa0\xa0The meat cost $7.00 per pound and the cheese cost $3.00 per pound.\xa0\xa0How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people?\xa0\n\xa0Student: To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.\nEach sandwich requires 1+1 = 2 pounds of meat and cheese.\nFor 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.\nThe cost of 10 pounds of meat is 10 x $7.00 = $70.\nThe cost of 10 pounds of cheese is 10 x $3.00 = $30.\nThe total cost of meat and cheese is $70 + $30 = $100.\n\xa0100\xa0\n\xa0Tutor: How many pounds o